# File: julia_periodicity_of_the_IDFT_and_DFT.ipynb
# This Julia script demonstrates the periodicity of the IDFT and DFT
AJ Wilkinson EEE3092F 2017-04; updated 2019-03-04 and 2020-03-04 and 2020-02-13 and 2023-02-16 and 2023-07-10

In [ ]:
#using Pkg
#Pkg.add("FFTW")
using FFTW       # Fourier library


In [ ]:
# Plots + Plotyly backend
# Plots defines a standard set of plot functions, and calls a backend plotting library
# Default backend is gr() [using the GR library]. 
# The plotly() backend allows zooming with a mouse in Jupyter Notebook.
# The pythonplot() backend uses PythonPlot (Python's matplotlib, which only allows zooming with the mouse if used from the REPL command line.
# Note: The initialisation time is irritatingly slow (as long as 1 minute), but thereafter it is quick to plot.
# Note2: In the code below, functions can be called via Plots.func() e.g. Plots.plot(), rather than just plot().  
# This allows one to use several plotting libraries at the same time. e.g.  One can also do PythonPlot.plot()

using Plots
plotly();    # Specify Plotly backend which allows zooming with a mouse inside Jupyter Notebook.

default(size=(700,400)); #Plot canvas size

default(label=""); # Turn off legends

default(ticks=:native);  #Ticks on x-axis are labelled nicely when zooming in.
   # ticks=:native Tells backend to calculate ticks by itself.
   # Good idea if you use interactive backends like plotly where you perform mouse zooming

default(grid=:false);  #Turn off grid
# default(linewidth=2)

# Specify font and canvas size defaults if you wish 
# fntsm = font("sans-serif", pointsize=round(10.0))
# fntlg = font("sans-serif", pointsize=round(14.0))
# default(titlefont=fntlg, guidefont=fntlg, tickfont=fntsm, legendfont=fntsm)

# Description of all adjustable Plots parameters:https://docs.juliaplots.org/latest/generated/attributes_series/

In [ ]:
# Create a sampled signal
x = [0,1,1,0,0,0,0,0];

# Caclulate the DFT using the FFT function
X = fft(x);

println("x = ",x);
println("abs.(X) = ",round.(abs.(X), digits=2));
println("angle.(X) = ",round.(angle.(X), digits=2));


In [ ]:
# Plot array x and the mag of X
N=length(x)

n=0:N-1  # Define range for plotting
k=0:N-1

fig1=scatter(n,x,ylabel="x[n]")
fig2=scatter(k,abs.(X),ylabel="abs X[k]")

fig=plot(fig1, fig2, layout = (2, 1), legend = false)
display(fig)

# Simple implementations of the DFT and inverse DFT functions from their definitions

In [ ]:
# Simple implementations of the DFT and inverse DFT functions from their definitions
# The DFT algorithm is O(N^2) and is inefficient compared to the FFT which is O(NlogN)
# These functions enable one to evaluate x[n] and X[n] outside the standard range of N values 
# Note in Julia and Matlab, arrays are indexed 1...N.
# In these functions, k and n are indexed 0...N-1 as in the mathematical definitions.


function dft(k,x)    # Calculate DFT for particular value of k where k is the freq index (usually in range k=0:N-1)
    N = length(x)
    X_k = zeros(length(k))
    for n=0:N-1
        X_k = X_k .+ x[n+1] * exp.(-im*2*pi*k*n/N)   # In Julia im == sqrt(-1)
    end
    return X_k
end

function idft(n,X)   # Calculate inverse DFT where n is the time index (usually in range 1:N)
    N = length(X)
    x_n = zeros(length(n))
    for k=0:N-1
        x_n = x_n .+ X[k+1] * exp.(im*2*pi*k*n/N)
    end
    x_n = x_n/N;
    return x_n
end


# Check to see if the dft() and idft() agree with the fft/ifft functions in the FFTW library 


In [ ]:

N = length(x)

k=0:N-1   # Calculate DFT for a range of k values

println( "dft(k,x) = ", (dft(k,x)) )
println( "fft(x) = ", X )


n=0:N-1   # Calculate IDFT for a range of n values
println("")
println( "x = ", x )
println( "real(idft(n,X)) = ", real(idft(n,X)) )


# Round to 3 significant figures
println("")
println( "dft(k,x) = ", round.(dft(k,x),digits=3) )
println( "fft(x) = ", X )

println("")
println( "x = ", x )
println( "real(idft(n,X)) = ", round.(idft(n,X),digits=3) )


# Illustrate periodicity in time and frequency domains


In [ ]:
# Plot results (points only)

n = -N:2*N-1
fig1 = scatter(n,real.(idft(n,X)),ylabel="Real idft(n,X)")

k = -N:2*N-1
fig2 = scatter(k,abs.(dft(k,x)),ylabel="Magnitude of dft(k,x)")

fig=plot(fig1, fig2, layout = (2, 1), legend = false)
display(fig)

# Try stem plots


In [ ]:
n = -N:2*N-1
fig1 = plot(n,real.(idft(n,X)),ylabel="Real idft(n,X)", line=:stem, marker=:circle, markersize=4)

k = -N:2*N-1
fig2=plot(k,abs.(dft(k,x)),ylabel="Magnitude of dft(k,x)", line=:stem, marker=:circle, markersize=4)

fig=plot(fig1, fig2, layout = (2, 1), legend = false)
display(fig)

# Plot results (points and lines joining points)


In [ ]:
n = -N:2*N-1
fig1 = plot(n,real.(idft(n,X)),ylabel="Real idft(n,X)", markershape = :circle, markersize = 4, markercolor=:lightblue, markerstrokecolor=:blue)

k = -N:2*N-1
fig2 = plot(k,abs.(dft(k,x)),ylabel="Magnitude of dft(k,x)", markershape = :circle, markersize = 4, markercolor=:lightblue, markerstrokecolor=:blue)

fig=plot(fig1, fig2, layout = (2, 1), legend = false)
display(fig)

# Demonstrating Discrete Convolution - from the definition


In [ ]:
# Create a sampled signal
x = [0,1,1,0,0,0,0,0];
y = [0,0.5,1,0,0,0,0,0]


N=length(x)

# Define a function to create the periodic extensions quickly
ext(n,x) = x[ mod.(n,length(x)) .+ 1]


default(size=(700,200)); #Plot canvas size
n=0:N-1   # Define for plotting
fig = plot(n,x, line=:stem, marker=:circle, markersize=4)
title!("Sequence x")
display(fig)

fig = plot(n,y, line=:stem, marker=:circle, markersize=4)
title!("Sequence y")
display(fig)


# Plot periodic extension of x
n = 0-2*N:2*N-1
fig = plot(n,ext(n,x), line=:stem, marker=:circle, markersize=4)
title!("Periodic extension of x")
display(fig)


# Plot periodic extension of y 
n = 0-2*N:2*N-1
fig = plot(n,ext(n,y), line=:stem, marker=:circle, markersize=4)
title!("Periodic extension of y")
display(fig)


# Plot flipped version of y
fig = plot(n,ext(-n,y), line=:stem, marker=:circle, markersize=4)
title!("y(-n)  i.e. flipped")
display(fig)


# Plot flipped and shifted version of y
m = 3  # Choose a shift
fig = plot(n,ext(m.-n,y), line=:stem, marker=:circle, markersize=4)
title!("y(m-n)  i.e. flipped and shifted by m=$m to the right")
display(fig)


# Plot final output of convolution  
# Quick method
n=0:N-1   # Define for plotting
z = real.(ifft( fft(x).*fft(y)))
fig = plot(n,z, line=:stem, marker=:circle, markersize=4)
title!("Discrete convolution result (0..N-1)")
display(fig)

n = 0-2*N:2*N-1
fig = plot(n,ext(n,z), line=:stem, marker=:circle, markersize=4)
title!("Discrete convolution result - periodic extension")
display(fig)


In [ ]:
# Testing 


ext(n,x) = x[ mod.(n,length(x)) .+ 1]
n = 0:N-1
fig = plot(n,x, line=:stem, marker=:circle, markersize=4)
display(fig)

n = 0-2*N:2*N-1
fig = plot(ext(0:2*N-1,x), line=:stem, marker=:circle, markersize=4)
display(fig)



# Demonstrating Discrete Convolution - using fast FFT-based method


In [ ]:
# Create a sampled signal
using FFTW

x = [0,1,1,0,0,0,0,0];
y = [0,0.5,1,0,0,0,0,0]

z = ifft(fft(x).*fft(y))

n=0:length(x)-1
fig = plot(n,x, line=:stem, marker=:circle, markersize=4)
title!("Sequence x")
display(fig)

fig = plot(n,y, line=:stem, marker=:circle, markersize=4)
title!("Sequence y")
display(fig)

fig = plot(n,real.(z), line=:stem, marker=:circle, markersize=4)
title!("Sequence z = x convolved with y")
display(fig)

# Zero padding in the time domain - Demo
For more examples, see file  julia_demo_of_Zero_padding_techniques.ipynb

In [ ]:

x = [0,1,1,0,0,0,0,0];

x_padded = [x; zeros(100)]

n=0:length(x)-1
fig = plot(n,x, line=:stem, marker=:circle, markersize=4)
title!("x")
display(fig)

n=0:length(x_padded)-1
fig = plot(n,x_padded, line=:stem, marker=:circle, markersize=2)
title!("x padded with zeros")
display(fig)

k=0:length(x)-1
fig = plot(k, abs.(fft(x)), line=:stem, marker=:circle, markersize=4)
title!("abs fft(x)")
display(fig)

k=0:length(x_padded)-1
fig = plot(k, abs.(fft(x_padded)), line=:stem, marker=:circle, markersize=2)
title!("abs fft(x padded with zeros)")
display(fig)
